Please provide the correct path to your `test.json` file. If the file is on your local machine, you can upload it to Colab by clicking the folder icon on the left sidebar, then the upload icon. Once uploaded, you can right-click on the file to copy its path.

Starts here


In [1]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Traceback (most recent call last):
  File "/usr/l

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM


In [3]:
!pip install -q transformers accelerate bitsandbytes sentencepiece huggingface_hub


In [4]:
from huggingface_hub import login
login()


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-2-7b-chat-hf"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Define 4-bit quantization config (W4A16)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load quantized model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

model.eval()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMS

In [46]:
def kg_prompt(text):
    return f"""
Extract all factual knowledge graph triples from the following sentence.

Return the output strictly in JSON format as an array of objects. Each object should have 'head', 'relation', and 'tail' keys.

Do not add any explanation or extra text. Only return valid JSON.

Sentence:
{text}
"""

def extract_triples(text):
    prompt = kg_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=0.0,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


In [47]:
relation_map = {
    "nationality": "/people/person/nationality",
    "place_of_birth": "/people/person/place_of_birth",
    "children": "/people/person/children",
    "capital": "/location/country/capital",
    "contains": "/location/location/contains",
    "place_of_death": "/people/deceased_person/place_of_death"
}

def normalize_relation(pred_rel):
    pred_rel = pred_rel.lower().strip()
    return relation_map.get(pred_rel, "/unknown/relation")


In [48]:
tail_map = {
    "usa": "united states",
    "u.s.": "united states",
    "us": "united states",
    # Add more as needed
}

def normalize_tail(tail):
    return tail_map.get(tail.lower().strip(), tail.lower().strip())


In [49]:
def parse_triples(output_text):
    try:
        json_part = re.search(r"\[.*\]", output_text, re.S).group()
        triples = json.loads(json_part)
        normalized_triples = []
        for t in triples:
            head = t.get("head", "").lower().strip()
            relation = normalize_relation(t.get("relation", ""))
            tail = normalize_tail(t.get("tail", ""))
            normalized_triples.append((head, relation, tail))
        return normalized_triples
    except Exception as e:
        return []


In [50]:
nyt_data = []
with open("/content/drive/MyDrive/FYP/Datasets/NYT-10 dataset/test.json", "r") as f:
    for line in f:
        nyt_data.append(json.loads(line))


In [51]:
def get_ground_truth_triples(sample):
    triples = []
    for r in sample.get("relationMentions", []):
        head = r.get("head", "").lower().strip()
        relation = r.get("label", "").lower().strip()
        tail = r.get("tail", "").lower().strip()
        relation = normalize_relation(relation)
        tail = normalize_tail(tail)
        triples.append((head, relation, tail))
    return triples


In [53]:
all_true_triples = []
all_pred_triples = []

for sample in nyt_data[:10]:
    text = sample["sentText"]
    gt_triples = get_ground_truth_triples(sample)
    all_true_triples.extend(gt_triples)

    output = extract_triples(text)
    pred_triples = parse_triples(output)

    if not pred_triples:
        pred_triples = [("","/unknown/relation","")]

    all_pred_triples.extend(pred_triples)

In [54]:
# Compute metrics
true_set = set(all_true_triples)
pred_set = set(all_pred_triples)

tp = len(true_set & pred_set)
fp = len(pred_set - true_set)
fn = len(true_set - pred_set)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Precision: 0.16666666666666666
Recall: 1.0
F1-score: 0.2857142857142857
